In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Simulation settings
SIMULATION_MINUTES = 60 * 24   # simulate 1 full day
TIME_STEP_SECONDS = 5           # check the intersection every 5 seconds

# Fixed signal timing (in seconds)
GREEN_TIME = 30
YELLOW_TIME = 3

# State variables
ns_queue = 0        # vehicles waiting on North-South
ew_queue = 0         # vehicles waiting on East-West
current_phase = "NS_GREEN"   # NS_GREEN, NS_YELLOW, EW_GREEN, EW_YELLOW
phase_timer = 0      # seconds spent in current phase

print("Simulation initialized.")
print(f"Will simulate {SIMULATION_MINUTES} minutes in steps of {TIME_STEP_SECONDS} seconds.")

Simulation initialized.
Will simulate 1440 minutes in steps of 5 seconds.


In [2]:
def get_arrival_rate(hour, direction):
    """
    Returns the average number of vehicles arriving per second
    for a given hour of day and direction (NS or EW).
    Mirrors the same peak pattern used in the ML dataset.
    """
    morning_peak = np.exp(-0.5 * ((hour - 8) / 1.5) ** 2)
    evening_peak = np.exp(-0.5 * ((hour - 17) / 2) ** 2)

    if direction == "NS":
        base = 8 + 35 * morning_peak + 30 * evening_peak
    else:  # EW
        base = 6 + 25 * morning_peak + 40 * evening_peak

    # base is vehicles per 5 minutes -> convert to vehicles per second
    vehicles_per_second = base / (5 * 60)
    return vehicles_per_second


# Quick sanity check: arrival rate at 8 AM vs 2 AM
print("NS arrival rate at 8 AM (per second):", get_arrival_rate(8, "NS"))
print("NS arrival rate at 2 AM (per second):", get_arrival_rate(2, "NS"))

NS arrival rate at 8 AM (per second): 0.14333733986307262
NS arrival rate at 2 AM (per second): 0.026705803973316313


In [3]:
# Reset state before running the full simulation
ns_queue = 0
ew_queue = 0
current_phase = "NS_GREEN"
phase_timer = 0

# Vehicles are removed from a green-direction queue at this rate (per second)
DISCHARGE_RATE_PER_SECOND = 0.5   # 1 vehicle every 2 seconds when green

# Records to track performance
records = []

total_steps = int((SIMULATION_MINUTES * 60) / TIME_STEP_SECONDS)

for step in range(total_steps):
    sim_time_seconds = step * TIME_STEP_SECONDS
    hour = (sim_time_seconds // 3600) % 24

    # 1. New vehicles arrive (probabilistically, based on time of day)
    ns_arrival_prob = get_arrival_rate(hour, "NS") * TIME_STEP_SECONDS
    ew_arrival_prob = get_arrival_rate(hour, "EW") * TIME_STEP_SECONDS

    if np.random.random() < ns_arrival_prob:
        ns_queue += 1
    if np.random.random() < ew_arrival_prob:
        ew_queue += 1

    # 2. Vehicles pass through if their direction has green
    if current_phase == "NS_GREEN":
        passed = min(ns_queue, DISCHARGE_RATE_PER_SECOND * TIME_STEP_SECONDS)
        ns_queue -= passed
    elif current_phase == "EW_GREEN":
        passed = min(ew_queue, DISCHARGE_RATE_PER_SECOND * TIME_STEP_SECONDS)
        ew_queue -= passed

    # 3. Advance the phase timer and switch phases on a fixed schedule
    phase_timer += TIME_STEP_SECONDS

    if current_phase == "NS_GREEN" and phase_timer >= GREEN_TIME:
        current_phase = "NS_YELLOW"
        phase_timer = 0
    elif current_phase == "NS_YELLOW" and phase_timer >= YELLOW_TIME:
        current_phase = "EW_GREEN"
        phase_timer = 0
    elif current_phase == "EW_GREEN" and phase_timer >= GREEN_TIME:
        current_phase = "EW_YELLOW"
        phase_timer = 0
    elif current_phase == "EW_YELLOW" and phase_timer >= YELLOW_TIME:
        current_phase = "NS_GREEN"
        phase_timer = 0

    # 4. Record the state at this step
    records.append({
        "sim_time_seconds": sim_time_seconds,
        "hour": hour,
        "phase": current_phase,
        "ns_queue": ns_queue,
        "ew_queue": ew_queue
    })

sim_df = pd.DataFrame(records)
print("Simulation complete.")
print(sim_df.shape)
sim_df.head(10)

Simulation complete.
(17280, 5)


,sim_time_seconds,hour,phase,ns_queue,ew_queue
0,0,0,NS_GREEN,0.0,0.0
1,5,0,NS_GREEN,0.0,0.0
2,10,0,NS_GREEN,0.0,0.0
3,15,0,NS_GREEN,0.0,0.0
4,20,0,NS_GREEN,0.0,0.0
5,25,0,NS_YELLOW,0.0,0.0
6,30,0,EW_GREEN,0.0,0.0
7,35,0,EW_GREEN,0.0,0.0
8,40,0,EW_GREEN,0.0,0.0
9,45,0,EW_GREEN,0.0,0.0


In [4]:
# Look at a busy period - around 8 AM (hour 8), roughly step 5760 onward
busy_period = sim_df[sim_df["hour"] == 8].head(20)
busy_period

,sim_time_seconds,hour,phase,ns_queue,ew_queue
5760,28800,8,EW_GREEN,1.0,5.0
5761,28805,8,EW_GREEN,2.0,3.5
5762,28810,8,EW_GREEN,3.0,1.0
5763,28815,8,EW_GREEN,4.0,0.0
5764,28820,8,EW_GREEN,5.0,0.0
5765,28825,8,EW_GREEN,5.0,0.0
5766,28830,8,EW_YELLOW,5.0,0.0
5767,28835,8,NS_GREEN,6.0,0.0
5768,28840,8,NS_GREEN,3.5,1.0
5769,28845,8,NS_GREEN,2.0,1.0


In [5]:
# Performance metrics for the fixed-time controller
avg_ns_queue = sim_df["ns_queue"].mean()
avg_ew_queue = sim_df["ew_queue"].mean()
max_ns_queue = sim_df["ns_queue"].max()
max_ew_queue = sim_df["ew_queue"].max()

print("Fixed-time controller performance (1 full day)")
print(f"Average North-South queue length: {avg_ns_queue:.2f}")
print(f"Average East-West queue length: {avg_ew_queue:.2f}")
print(f"Maximum North-South queue length: {max_ns_queue:.2f}")
print(f"Maximum East-West queue length: {max_ew_queue:.2f}")

Fixed-time controller performance (1 full day)
Average North-South queue length: 0.97
Average East-West queue length: 0.90
Maximum North-South queue length: 8.00
Maximum East-West queue length: 8.00


In [6]:
# Peak-hour only (8 AM) performance — more revealing than the whole-day average
peak_hour_data = sim_df[sim_df["hour"] == 8]

print("Fixed-time controller performance (8 AM peak hour only)")
print(f"Average North-South queue length: {peak_hour_data['ns_queue'].mean():.2f}")
print(f"Average East-West queue length: {peak_hour_data['ew_queue'].mean():.2f}")
print(f"Maximum North-South queue length: {peak_hour_data['ns_queue'].max():.2f}")
print(f"Maximum East-West queue length: {peak_hour_data['ew_queue'].max():.2f}")

Fixed-time controller performance (8 AM peak hour only)
Average North-South queue length: 2.43
Average East-West queue length: 1.61
Maximum North-South queue length: 8.00
Maximum East-West queue length: 7.00
